### Feature engineering.

In [22]:
import pandas as pd
import numpy as np

In [23]:
df_territorial = pd.read_csv("CSVs/df_turismo_limpio.csv")

De la columna "Tipo_turista" se filtran las filas correspondientes a "Viajero".

In [24]:
df_viajeros = df_territorial[df_territorial["Tipo_turista"] == "Viajero"].copy()

Se agrupa por provincia, mes y año.

In [25]:
df_viajeros_totales = df_viajeros.groupby(
    ["Provincias", "Año", "Mes"],
    as_index=False
).agg({"Turistas_totales" : "sum"})

Se usan solo los meses de verano.

In [26]:
verano_viajeros = [6,7,8]
df_verano_viajeros = df_viajeros_totales[df_viajeros_totales["Mes"].isin(verano_viajeros)].copy()

df_verano_viajeros = df_verano_viajeros.sort_values(["Provincias", "Año", "Mes"]).reset_index(drop=True)

Se crea un dataframe con nuevas filas que contendrán el resultadod e las predicciones del año 2026.

In [27]:
provincias_unicas = df_verano_viajeros["Provincias"].unique()
meses_2026 = [6,7,8]
año = 2026

filas_predicciones = []

for prov in provincias_unicas:
    for mes in meses_2026:
        filas_predicciones.append({
            "Provincias" : prov,
            "Año" : año,
            "Mes" : mes,
            "Turistas_totales" : np.nan
        })

df_predicciones = pd.DataFrame(filas_predicciones)

df_verano_viajeros = pd.concat([df_verano_viajeros, df_predicciones], ignore_index=True)

df_verano_viajeros = df_verano_viajeros.sort_values(["Provincias", "Año", "Mes"]).reset_index(drop=True)

Se crean lags.

In [28]:
grupo_provincias = df_verano_viajeros.groupby("Provincias")

df_verano_viajeros["Turistas_pasados"] = df_verano_viajeros.groupby("Provincias")["Turistas_totales"].shift(1)
df_verano_viajeros["lag_estacional"] = df_verano_viajeros.groupby("Provincias")["Turistas_totales"].shift(3)

for lag in [1,2,3,4]:
    df_verano_viajeros[f"lag_{lag}"] = grupo_provincias["Turistas_totales"].shift(lag)

Se establece "log_ratio" como target.

In [29]:
df_verano_viajeros["log_ratio"] = np.log(df_verano_viajeros["Turistas_totales"].clip(lower=1)) - np.log(df_verano_viajeros["Turistas_pasados"].clip(lower=1))

Transformación de los lags a escala logarítima para hacer posible el modelado en consonancia con el target.

In [30]:
for lag in [1,2,3,4]:
    df_verano_viajeros[f"log_lag_{lag}"] = np.log1p(df_verano_viajeros[f"lag_{lag}"])

Feature principal.

In [31]:
df_verano_viajeros["log_lag_estacional"] = np.log1p(df_verano_viajeros["lag_estacional"])

Nuevas variables de tendencia.

In [32]:
df_verano_viajeros["media_lag_2"] = (df_verano_viajeros["log_lag_1"] + df_verano_viajeros["log_lag_2"]) / 2

df_verano_viajeros["media_lag_3"] = (df_verano_viajeros["log_lag_1"] +
                                    df_verano_viajeros["log_lag_2"] +
                                    df_verano_viajeros["log_lag_3"]) / 3

df_verano_viajeros["tendencia_corta"] = df_verano_viajeros["log_lag_1"] - df_verano_viajeros["log_lag_2"]

Limpieza final.

In [33]:
df_verano_limpio = df_verano_viajeros.dropna(subset=["lag_estacional"]).copy()

Se separa train y test y se aplica una condición para filtrar los años del covid.

In [34]:
anios_covid = [2020, 2021]
anio_inicio_train = 2018

condicion_covid = (
    (df_verano_limpio["Año"] < 2025) & (df_verano_limpio["Año"].isin(anios_covid) == False) & (df_verano_limpio["Año"] >= anio_inicio_train))

train_viajeros = df_verano_limpio[condicion_covid].copy()

In [35]:
test_viajeros = df_verano_viajeros[df_verano_viajeros["Año"] == 2025].copy()

Se prepara 2026.

In [36]:
prediccion_viajeros = df_verano_viajeros[df_verano_viajeros["Año"] == 2026]

Exporto CSVs.

In [37]:
train_viajeros.to_csv("CSVs/train_viajeros.csv", index= False)
test_viajeros.to_csv("CSVs/test_viajeros.csv", index=False)
prediccion_viajeros.to_csv("CSVs/prediccion_viajeros.csv", index=False)
df_verano_viajeros.to_csv("CSVs/verano_viajeros.csv", index=False)
df_viajeros_totales.to_csv("CSVs/viajeros_totales.csv", index=False)